# 📘 Module 06 — PyTorch from Scratch
## Chapter 6.1 — Tensors: The Universal Currency of Deep Learning

> **Learning Path:** Zero to Hero — AI Engineer  
> **Module:** 06 of 18  
> **Topics:** Dimensional Hierarchy & Memory Strides, Creation & Zero-Copy NumPy Bridging, Arithmetic & Broadcasting, Device Routing (CPU/CUDA/MPS), Dimension Surgery (.view vs .reshape, squeeze, unsqueeze)  
> **Format:** Notes first → Code second  

---


## 🎯 What You Will Learn

By the end of this chapter, you will be able to:

- [ ] Explain tensors as multidimensional strided memory arrays and contrast them with nested Python lists
- [ ] Create tensors using factory functions (`zeros`, `ones`, `randn`) and zero-copy NumPy bridging (`from_numpy`)
- [ ] Execute tensor arithmetic, broadcasting, and matrix multiplication (`@`) with strict dimension verification
- [ ] Dynamically route tensors between CPU, CUDA GPUs, and Apple Silicon MPS hardware accelerators
- [ ] Master dimension surgery: navigate contiguous memory layout, `.view()` vs `.reshape()`, `squeeze()`, and `unsqueeze()`

**Estimated Reading Time:** ~35 minutes


## 🗂️ Table of Contents

1. [What is a Tensor? — Dimensional Hierarchy & Memory Storage](#1-what-is-a-tensor--dimensional-hierarchy--memory-storage)
2. [Tensor Creation & Zero-Copy NumPy Bridging](#2-tensor-creation--zero-copy-numpy-bridging)
3. [Tensor Arithmetic — Element-wise, Broadcasting, & Matrix Multiplication](#3-tensor-arithmetic--element-wise-broadcasting--matrix-multiplication)
4. [Device Management — CPU vs GPU vs Apple Silicon MPS](#4-device-management--cpu-vs-gpu-vs-apple-silicon-mps)
5. [Dimension Surgery — View vs Reshape, Squeeze, & Unsqueeze](#5-dimension-surgery--view-vs-reshape-squeeze--unsqueeze)
6. [Chapter Summary](#-chapter-summary)
7. [Resources & Further Reading](#-resources--further-reading)
8. [Next Chapter Preview](#-next-chapter-chapter-62--autograd-pytorchs-automatic-differentiation)


---

## 1. What is a Tensor? — Dimensional Hierarchy & Memory Storage

---

### 🗣️ In Simple English

A **Tensor** is simply a container for numbers organized into a grid with any number of dimensions:
- A single number is a **0-Dimensional Tensor** (a Scalar). Example: Loss $= 0.42$.
- A list of numbers is a **1-Dimensional Tensor** (a Vector). Example: Clinical biomarkers of a patient $= [34, 120, 85]$.
- A spreadsheet table of rows and columns is a **2-Dimensional Tensor** (a Matrix). Example: A batch of patients $\times$ biomarkers.
- A stack of matrices is a **3-Dimensional Tensor** (a Cube). Example: Sequential sentences in text $= [\text{Batch}, \text{Tokens}, \text{Embedding}]$.
- A collection of color images is a **4-Dimensional Tensor**. Example: $[\text{Batch}, \text{Channels}, \text{Height}, \text{Width}]$.

The key point is: No matter how many dimensions a tensor appears to have in Python, inside your computer's RAM or GPU VRAM, it is stored as **one single flat, contiguous 1D array of memory bytes**! PyTorch keeps track of the multidimensional shape using a secret offset system called **Strides**.

---

### 🌍 Real-Life Example

**Scenario:** Representing diverse sensory data modalities inside an autonomous vehicle perception system.

| Modality in Autonomous Car | Tensor Rank | Tensor Dimensions (Shape) | Physical Meaning |
|:---|:---:|:---:|:---|
| **Vehicle Speed** | 0D (Scalar) | `()` | Scalar speedometer reading: $65.4\text{ km/h}$ |
| **Radar Distance Array** | 1D (Vector) | `(64,)` | 64 range measurements from ultrasonic bumper sensors |
| **Tabular Vehicle Telemetry** | 2D (Matrix) | `(32, 12)` | Batch of 32 time steps, each with 12 engine telemetry channels |
| **Audio Speech Commands** | 3D (Volume) | `(8, 1, 16000)` | 8 audio clips, 1 microphone channel, 16,000 audio samples (1 second) |
| **Camera Vision Feeds** | 4D (Batch) | `(16, 3, 720, 1280)` | Batch of 16 dashboard camera frames: RGB channels, $720 \times 1280$ resolution |
| **LiDAR Point Clouds / Video** | 5D (Spatio-temporal) | `(4, 30, 3, 224, 224)` | 4 video clips, 30 video frames per clip, RGB channels, $224 \times 224$ pixels |

---

### 📖 How It Works (Step by Step)

#### Storage and Strides (The Hardware Reality)
How does PyTorch convert a 2D matrix of shape `(3, 4)` into hardware memory?
1. **Flat Storage (`Storage`):** PyTorch allocates 12 consecutive floating-point slots in RAM:
   `[x00, x01, x02, x03, x10, x11, x12, x13, x20, x21, x22, x23]`
2. **The Stride Tuple (`stride`):** Strides dictate how many memory slots you must jump across to advance by $1$ in each dimension.
   - For a `(3, 4)` matrix: `stride = (4, 1)`.
   - To jump down to the next row (dimension 0), skip forward **4 slots**.
   - To jump to the next column (dimension 1), skip forward **1 slot**.
   - Formula for coordinate `(i, j)`: $\text{Memory Index} = i \times \text{stride}[0] + j \times \text{stride}[1] = i \times 4 + j \times 1$.

---

### 🔑 Key Concepts

| Concept | Plain English Meaning | Why It Matters |
|:---|:---|:---|
| **Rank (Dimension count)** | Number of axes in the tensor (`tensor.ndim`). | Dictates the structural complexity of data. |
| **Shape** | Number of elements along each axis (`tensor.shape`). | Must match dimension rules during matrix multiplications. |
| **Dtype (Data Type)** | Bit-depth and format of numbers (`float32`, `int64`, etc.). | Controls numerical precision, memory footprint, and GPU speed. |
| **Stride** | Step-sizes in flat memory required to traverse each dimension. | Allows zero-copy transpositions and slicing! |

---

### 🗺️ Visual Overview

```
THE TENSOR RANK HIERARCHY:

Rank 0 (Scalar)        Rank 1 (Vector)         Rank 2 (Matrix)           Rank 3 (3D Tensor)
    [ 42 ]             [ 1, 2, 3, 4 ]          ┌───┬───┬───┐             ┌───┬───┐
                                               ├───┼───┼───┤            /───/───/│
                                               └───┴───┴───┘           ┌───┬───┐││
                                                                       ├───┼───┤│/
                                                                       └───┴───┘/
  Shape: ()             Shape: (4,)             Shape: (3, 3)          Shape: (2, 2, 2)
```

---

### 🚫 Common Misconceptions

| ❌ Myth | ✅ Reality |
|:---|:---|
| *"Transposing a matrix copies all its data into a newly allocated block of memory."* | **False!** In PyTorch, transposing `tensor.t()` simply **swaps the strides** (`(4, 1) -> (1, 4)`). Zero numbers are moved in memory, making transpositions virtually instantaneous ($O(1)$ time complexity)! |

---

### ✅ When to Use This

- Always inspect `.shape`, `.dtype`, and `.device` whenever debugging PyTorch errors.

---

### 📊 Section Quick Reference

| Attribute | PyTorch Syntax | Typical Value |
|:---|:---|:---:|
| **Shape** | `t.shape` or `t.size()` | `torch.Size([32, 3, 224, 224])` |
| **Number of Dimensions** | `t.ndim` or `t.dim()` | `4` |
| **Total Number of Elements** | `t.numel()` | $32 \times 3 \times 224 \times 224 = 4,816,896$ |
| **Memory Strides** | `t.stride()` | `(150528, 50176, 224, 1)` |
| **Data Type** | `t.dtype` | `torch.float32` |

---


In [1]:
# ✅ Code Demo: Inspecting Tensor Dimensionality, Memory Strides, and Data Types
# Concept: Scalars to 4D Batches, Flat Storage, and Stride Arithmetic
# -----------------------------------------------------------------------------

import os
import torch
import numpy as np

# Ensure local asset path exists
assets_dir = 'assets' if os.path.exists('assets') else 'Module-06-PyTorch-Foundations/assets'
os.makedirs(assets_dir, exist_ok=True)

# 1. Creating Tensors across the Rank Hierarchy
scalar_0d = torch.tensor(3.14159)
vector_1d = torch.tensor([10.0, 20.0, 30.0, 40.0])
matrix_2d = torch.tensor([
    [1, 2, 3, 4],
    [5, 6, 7, 8],
    [9, 10, 11, 12]
], dtype=torch.float32)
volume_3d = torch.randn(2, 3, 4)            # e.g., (Batch=2, Sequence=3, Features=4)
batch_4d  = torch.randn(8, 3, 32, 32)       # e.g., (Batch=8, RGB=3, Height=32, Width=32)

print("=" * 70)
print("PYTORCH TENSOR RANK & SHAPE INSPECTION")
print("=" * 70)
ranks = [
    ("0D Scalar", scalar_0d),
    ("1D Vector", vector_1d),
    ("2D Matrix", matrix_2d),
    ("3D Volume", volume_3d),
    ("4D Vision Batch", batch_4d)
]

for name, t in ranks:
    print(f"{name:<18} | Shape: {str(list(t.shape)):<18} | Dtype: {str(t.dtype):<15} | Numel: {t.numel():<7}")
print("=" * 70)

# 2. Exploring Strides and Memory Storage
print("\nUNDER THE HOOD: FLAT STORAGE & STRIDES")
print("-" * 70)
print(f"2D Matrix Shape:       {matrix_2d.shape}")
print(f"2D Matrix Strides:     {matrix_2d.stride()}")
print(f"Element at row 2, col 1: {matrix_2d[2, 1].item()}")
# Calculate manual flat index: row * stride[0] + col * stride[1]
flat_idx = 2 * matrix_2d.stride()[0] + 1 * matrix_2d.stride()[1]
print(f"Manual Flat Index:     2 * {matrix_2d.stride()[0]} + 1 * {matrix_2d.stride()[1]} = {flat_idx}")
# Modern PyTorch uses untyped_storage() or flatten()
print(f"Value in Flat Storage: {matrix_2d.untyped_storage()[flat_idx * 4]} (Float32 is 4 bytes)")
print(f"Value via Flatten:     {matrix_2d.flatten()[flat_idx].item()}")
print("-" * 70)


PYTORCH TENSOR RANK & SHAPE INSPECTION
0D Scalar          | Shape: []                 | Dtype: torch.float32   | Numel: 1      
1D Vector          | Shape: [4]                | Dtype: torch.float32   | Numel: 4      
2D Matrix          | Shape: [3, 4]             | Dtype: torch.float32   | Numel: 12     
3D Volume          | Shape: [2, 3, 4]          | Dtype: torch.float32   | Numel: 24     
4D Vision Batch    | Shape: [8, 3, 32, 32]     | Dtype: torch.float32   | Numel: 24576  

UNDER THE HOOD: FLAT STORAGE & STRIDES
----------------------------------------------------------------------
2D Matrix Shape:       torch.Size([3, 4])
2D Matrix Strides:     (4, 1)
Element at row 2, col 1: 10.0
Manual Flat Index:     2 * 4 + 1 * 1 = 9
Value in Flat Storage: 0 (Float32 is 4 bytes)
Value via Flatten:     10.0
----------------------------------------------------------------------


---

## 2. Tensor Creation & Zero-Copy NumPy Bridging

---

### 🗣️ In Simple English

How do you create tensors in PyTorch?
1. **From Scratch:** You can initialize blank tensors filled with zeros (`torch.zeros`), ones (`torch.ones`), uniform random noise (`torch.rand`), or Gaussian bell-curve noise (`torch.randn`).
2. **From Existing Data:** You can convert Python lists directly into tensors (`torch.tensor([1, 2, 3])`).
3. **The NumPy Bridge:** You can convert NumPy arrays to PyTorch tensors with `torch.from_numpy(arr)` and convert back with `tensor.numpy()`.

The crucial architectural detail is **Zero-Copy Memory Sharing**.
When you convert a NumPy array into a PyTorch CPU tensor using `torch.from_numpy()`, PyTorch does **NOT duplicate the data in RAM**! Both NumPy and PyTorch point to the **exact same memory address**. If you change a number in the NumPy array, the PyTorch tensor changes instantly, and vice versa!

---

### 🌍 Real-Life Example

**Scenario:** High-throughput computer vision pipeline ingesting 4K video frames from an OpenCV camera driver.

| Operation Step | Without Zero-Copy (Slow) | With Zero-Copy `torch.from_numpy` (Fast) |
|:---|:---|:---|
| **4K Video Frame ($3840 \times 2160 \times 3$)** | Copies $25\text{ MB}$ of pixel data to a new memory buffer | **Zero bytes copied!** PyTorch simply wraps a tensor header around the OpenCV buffer |
| **Processing 60 FPS** | Allocates and copies $1.5\text{ GB}$ of RAM every second | **Zero memory allocation latency**; CPU cache remains warm |

---

### 📖 How It Works (Step by Step)

#### Factory Functions Comparison
1. `torch.zeros(shape)`: Filled with $0.0$.
2. `torch.ones(shape)`: Filled with $1.0$.
3. `torch.empty(shape)`: Allocates raw memory without initializing values (contains whatever random garbage was in RAM; fastest allocation!).
4. `torch.rand(shape)`: Uniform random numbers sampled from $\mathcal{U}[0, 1)$.
5. `torch.randn(shape)`: Standard normal Gaussian noise sampled from $\mathcal{N}(0, 1)$ (mean $0$, variance $1$).
6. `torch.arange(start, end, step)`: Evenly spaced integers or floats.
7. `torch.linspace(start, end, steps)`: Fixed number of evenly spaced points.

---

### 🔑 Key Concepts

| Method | Memory Behavior | When to Use |
|:---|:---|:---|
| `torch.tensor(data)` | **Copies data** into newly allocated memory. | When you want an independent copy from a Python list or NumPy array. |
| `torch.from_numpy(arr)` | **Zero-copy:** Shares the exact same memory buffer. | When bridging NumPy preprocessing pipelines to PyTorch without latency. |
| `t.numpy()` | **Zero-copy:** Exposes tensor memory as a NumPy array. | When moving model predictions to Matplotlib, Pandas, or Scikit-Learn. |
| `t.clone()` | Performs a deep copy of tensor data and computation history. | When you need an independent tensor copy that retains autograd connections. |

---

### 🗺️ Visual Overview

```
THE ZERO-COPY NUMPY <-> PYTORCH SHARED MEMORY BRIDGE:

   NumPy Array Pointer (arr) ──────┐
                                   ▼
                   ┌───────────────────────────────────┐
                   │ Memory Address 0x7ffd98b0:        │
                   │ [ 1.0 , 2.0 , 3.0 , 4.0 , 5.0 ]   │
                   └───────────────────────────────────┘
                                   ▲
   PyTorch Tensor Pointer (t) ─────┘
   Modifying arr[0] = 99.0 instantly changes t[0] to 99.0 with zero memory copying!
```

---

### 🚫 Common Misconceptions

| ❌ Myth | ✅ Reality |
|:---|:---|
| *"Can I call `t.numpy()` on a tensor located on my GPU?"* | **No!** NumPy does not support GPU memory. If a tensor is on CUDA, calling `t.numpy()` throws `RuntimeError: Tensor on device cuda:0 is not on CPU!`. You must call `t.cpu().numpy()`! |

---

### ✅ When to Use This

- Use `torch.randn` for weight initialization (standard normal distribution).
- Use `torch.from_numpy` when feeding OpenCV images or Pandas DataFrames into PyTorch models.

---

### 📊 Section Quick Reference

| Syntax | Description | Example |
|:---|:---|:---|
| `torch.zeros(2, 3)` | Matrix of zeros | `[[0., 0., 0.], [0., 0., 0.]]` |
| `torch.randn(2, 3)` | Normal random matrix | Gaussian noise $\mu=0, \sigma=1$ |
| `torch.from_numpy(arr)` | Zero-copy NumPy conversion | Shared underlying storage |
| `t.clone().detach()` | Complete independent snapshot | Safe detachment from computational graph |

---


In [2]:
# ✅ Code Demo: Tensor Factory Functions and Zero-Copy NumPy Memory Sharing
# Concept: Initializing Tensors and Proving Shared In-Place Memory Modification
# ---------------------------------------------------------------------------

import os
import torch
import numpy as np

# Ensure local asset path exists
assets_dir = 'assets' if os.path.exists('assets') else 'Module-06-PyTorch-Foundations/assets'
os.makedirs(assets_dir, exist_ok=True)

# 1. Essential Factory Functions
torch.manual_seed(42)
t_zeros = torch.zeros((2, 3))
t_ones  = torch.ones((2, 3))
t_rand  = torch.rand((2, 3))    # Uniform [0, 1)
t_randn = torch.randn((2, 3))   # Normal N(0, 1)
t_range = torch.arange(0, 10, 2)

print("=" * 65)
print("PYTORCH FACTORY FUNCTIONS DEMONSTRATION")
print("=" * 65)
print(f"1. torch.zeros(2,3):\n{t_zeros}")
print(f"\n2. torch.randn(2,3) [Mean={t_randn.mean():.2f}, Std={t_randn.std():.2f}]:\n{t_randn.round(decimals=3)}")
print(f"\n3. torch.arange(0, 10, 2): {t_range}")
print("-" * 65)

# 2. Demonstrating the Zero-Copy NumPy Bridge
print("PROOF OF ZERO-COPY MEMORY SHARING (NumPy <-> PyTorch)")
print("-" * 65)

# Create a NumPy array
np_arr = np.array([10.0, 20.0, 30.0, 40.0])
# Bridge to PyTorch without copying memory
pt_tensor = torch.from_numpy(np_arr)

print(f"Original NumPy Array:  {np_arr}")
print(f"Bridged PyTorch Tensor: {pt_tensor}")

# Mutate the NumPy array in-place
np_arr[0] = 999.0
print("\n[Action]: Mutated np_arr[0] = 999.0")
print(f"Updated NumPy Array:   {np_arr}")
print(f"Updated PyTorch Tensor: {pt_tensor}  <-- Automatically changed without copying!")

# Mutate the PyTorch tensor in-place
pt_tensor[3] = -777.0
print("\n[Action]: Mutated pt_tensor[3] = -777.0")
print(f"Updated NumPy Array:   {np_arr}  <-- Automatically reflected in NumPy!")
print(f"Updated PyTorch Tensor: {pt_tensor}")
print("=" * 65)


PYTORCH FACTORY FUNCTIONS DEMONSTRATION
1. torch.zeros(2,3):
tensor([[0., 0., 0.],
        [0., 0., 0.]])

2. torch.randn(2,3) [Mean=-0.06, Std=1.25]:
tensor([[ 1.1560,  0.3960, -2.4660],
        [ 0.3620,  0.3770, -0.1810]])

3. torch.arange(0, 10, 2): tensor([0, 2, 4, 6, 8])
-----------------------------------------------------------------
PROOF OF ZERO-COPY MEMORY SHARING (NumPy <-> PyTorch)
-----------------------------------------------------------------
Original NumPy Array:  [10. 20. 30. 40.]
Bridged PyTorch Tensor: tensor([10., 20., 30., 40.], dtype=torch.float64)

[Action]: Mutated np_arr[0] = 999.0
Updated NumPy Array:   [999.  20.  30.  40.]
Updated PyTorch Tensor: tensor([999.,  20.,  30.,  40.], dtype=torch.float64)  <-- Automatically changed without copying!

[Action]: Mutated pt_tensor[3] = -777.0
Updated NumPy Array:   [ 999.   20.   30. -777.]  <-- Automatically reflected in NumPy!
Updated PyTorch Tensor: tensor([ 999.,   20.,   30., -777.], dtype=torch.float64)


---

## 3. Tensor Arithmetic — Element-wise, Broadcasting, & Matrix Multiplication

---

### 🗣️ In Simple English

How do tensors interact mathematically?
1. **Element-wise Math:** Operations like `+`, `-`, `*`, `/`, and `**` operate on matching positions independently.
2. **Matrix Multiplication (`@` or `torch.matmul`):** The true engine of neural networks. It computes dot products of rows and columns ($Z = XW$).
3. **Broadcasting:** When you add a tiny 1D bias vector to a massive 2D matrix, PyTorch does not throw an error. It automatically "stretches" the smaller tensor across the missing dimension without wasting memory!

The fatal beginner trap is: **`A * B` is NOT matrix multiplication!**
- `A * B` is **element-wise multiplication** (Hadamard product). Both tensors must have matching shapes.
- `A @ B` (or `torch.matmul(A, B)`) is **true linear algebra matrix multiplication**!

---

### 🌍 Real-Life Example

**Scenario:** Adding layer bias offsets to a batch of 128 images during inference.

Suppose a convolutional layer outputs feature maps of shape `(128, 64, 28, 28)` (128 images, 64 feature channels, $28 \times 28$ pixels), and we want to add a channel bias vector of shape `(64, 1, 1)`.

| Tensor | Shape | Role in Convolutional Layer |
|:---|:---:|:---|
| **Feature Tensor ($A$)** | `(128, 64, 28, 28)` | Activations extracted by 64 convolution filters |
| **Bias Tensor ($B$)** | `(64, 1, 1)` | 64 learned threshold constants (one per channel) |
| **Result ($A + B$)** | `(128, 64, 28, 28)` | **Broadcasting:** The 64 biases are replicated across all 128 images and all $28 \times 28$ spatial pixels instantly! |

---

### 📖 How It Works (Step by Step)

#### A. The Broadcasting Rules
Two tensors can be broadcast together if, starting from their **trailing (rightmost) dimensions** and working backwards:
1. The dimension sizes are **equal**, OR
2. One of the dimension sizes is **$1$**, OR
3. One of the dimensions does **not exist**.

*Example:*
- Tensor A: `(5, 1, 4)`
- Tensor B: `   (3, 4)`
- Result:   `(5, 3, 4)` ✅ (Dimension 1 expands from 1 to 3; Dimension 0 expands from 1 to 5).

#### B. Matrix Multiplication Variants in PyTorch
1. `torch.mm(A, B)`: Strict 2D matrix multiplication only. Throws an error if tensors are 1D or 3D+.
2. `torch.bmm(A, B)`: Batched 3D matrix multiplication: `(B, N, M) @ (B, M, P) -> (B, N, P)`.
3. `A @ B` or `torch.matmul(A, B)`: The universal, omnipotent matrix multiplication function. Supports 1D, 2D, and arbitrary multidimensional batching (used in Transformers)!

---

### 🔑 Key Concepts

| Operator | Math Notation | What It Does | Example Shapes |
|:---|:---:|:---|:---:|
| `A * B` | $A \odot B$ | Element-wise Hadamard multiplication | `(3, 4) * (3, 4) -> (3, 4)` |
| `A @ B` | $A \times B$ | Linear Algebra Dot Product Matrix Multiply | `(3, 4) @ (4, 5) -> (3, 5)` |
| `A.t()` | $A^T$ | Transposes 2D matrix | `(3, 4).t() -> (4, 3)` |
| `A.permute(0, 2, 3, 1)` | Axis reordering | Reorders arbitrary dimensions (e.g., PyTorch NCHW $\to$ Matplotlib NHWC) | `(B, C, H, W) -> (B, H, W, C)` |

---

### 🗺️ Visual Overview

```
ELEMENT-WISE (A * B)   VS   MATRIX MULTIPLICATION (A @ B):

Element-wise Multiplication:             Matrix Multiplication:
┌───┬───┐       ┌───┬───┐   ┌───────┬───────┐      ┌───┬───┐       ┌───┬───┐   ┌───────┬───────┐
│ 1 │ 2 │   *   │ 5 │ 6 │ = │ 1*5=5 │ 2*6=12│      │ 1 │ 2 │   @   │ 5 │ 6 │ = │1*5+2*7│1*6+2*8│
├───┼───┤       ├───┼───┤   ├───────┼───────┤      ├───┼───┤       ├───┼───┤   ├───────┼───────┤
│ 3 │ 4 │       │ 7 │ 8 │   │ 3*7=21│ 4*8=32│      │ 3 │ 4 │       │ 7 │ 8 │   │3*5+4*7│3*6+4*8│
└───┴───┘       └───┴───┘   └───────┴───────┘      └───┴───┘       └───┴───┘   └───────┴───────┘
```

---

### 🚫 Common Misconceptions

| ❌ Myth | ✅ Reality |
|:---|:---|
| *"In deep learning, we use `torch.dot()` to multiply matrices."* | **A common trap!** In PyTorch, `torch.dot(x, y)` **only works on 1D vectors** and returns a scalar! For 2D matrices, you must use `torch.mm` or the universal `@` operator (`torch.matmul`). |

---

### ✅ When to Use This

- Always use the `@` operator for clean, readable linear layer matrix multiplications ($X @ W$).
- Use `.permute()` when converting PyTorch images `(Channels, Height, Width)` to Matplotlib `(Height, Width, Channels)`.

---

### 📊 Section Quick Reference

| Operation | Syntax | Key Rule |
|:---|:---|:---|
| **Element-wise Mult** | `A * B` | Shapes must match or broadcast |
| **Matrix Multiply** | `A @ B` | Inner dimensions must match: $(N, K) \times (K, M) = (N, M)$ |
| **In-Place Operations** | `A.add_(B)` or `A += B` | Trailing underscore `_` denotes in-place memory mutation |

---


In [3]:
# ✅ Code Demo: Element-wise Arithmetic vs Matrix Multiplication and Permutation
# Concept: Hadamard Product, Matmul Operator (@), Broadcasting, and Axis Permuting
# ---------------------------------------------------------------------------------

import os
import torch
import numpy as np

# Ensure local asset path exists
assets_dir = 'assets' if os.path.exists('assets') else 'Module-06-PyTorch-Foundations/assets'
os.makedirs(assets_dir, exist_ok=True)

# 1. Element-wise (*) vs Matrix Multiplication (@)
A = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
B = torch.tensor([[5.0, 6.0], [7.0, 8.0]])

elem_mult = A * B
matmul_res = A @ B

print("=" * 65)
print("ELEMENT-WISE (*) VS MATRIX MULTIPLICATION (@)")
print("=" * 65)
print(f"Matrix A (2x2):\n{A}")
print(f"\nMatrix B (2x2):\n{B}")
print(f"\n1. Element-wise (A * B) [Hadamard]:\n{elem_mult}")
print(f"\n2. Matrix Multiplication (A @ B) [Dot Products]:\n{matmul_res}")
print("-" * 65)

# 2. Broadcasting in Action: Adding Bias Vector (1, 3) to Batch (4, 3)
X_batch = torch.ones((4, 3)) * 10.0
bias_vec = torch.tensor([[1.0, 2.0, 3.0]])  # Shape: (1, 3)
broadcasted_sum = X_batch + bias_vec

print(f"Batch X Shape:     {X_batch.shape}")
print(f"Bias Vector Shape: {bias_vec.shape}")
print(f"Broadcasted Sum Shape: {broadcasted_sum.shape}")
print(f"Resulting Batch Values:\n{broadcasted_sum}")
print("-" * 65)

# 3. Axis Permutation: Converting PyTorch NCHW format to Matplotlib NHWC format
# Simulating a batch of 2 images: (Batch=2, Channels=3, Height=4, Width=4)
images_nchw = torch.randn(2, 3, 4, 4)
images_nhwc = images_nchw.permute(0, 2, 3, 1)

print("AXIS REORDERING VIA PERMUTE:")
print(f"  PyTorch Standard (NCHW):    {images_nchw.shape}")
print(f"  Matplotlib Standard (NHWC): {images_nhwc.shape}")
print("=" * 65)


ELEMENT-WISE (*) VS MATRIX MULTIPLICATION (@)
Matrix A (2x2):
tensor([[1., 2.],
        [3., 4.]])

Matrix B (2x2):
tensor([[5., 6.],
        [7., 8.]])

1. Element-wise (A * B) [Hadamard]:
tensor([[ 5., 12.],
        [21., 32.]])

2. Matrix Multiplication (A @ B) [Dot Products]:
tensor([[19., 22.],
        [43., 50.]])
-----------------------------------------------------------------
Batch X Shape:     torch.Size([4, 3])
Bias Vector Shape: torch.Size([1, 3])
Broadcasted Sum Shape: torch.Size([4, 3])
Resulting Batch Values:
tensor([[11., 12., 13.],
        [11., 12., 13.],
        [11., 12., 13.],
        [11., 12., 13.]])
-----------------------------------------------------------------
AXIS REORDERING VIA PERMUTE:
  PyTorch Standard (NCHW):    torch.Size([2, 3, 4, 4])
  Matplotlib Standard (NHWC): torch.Size([2, 4, 4, 3])


---

## 4. Device Management — CPU vs GPU vs Apple Silicon MPS

---

### 🗣️ In Simple English

Every tensor lives in a specific hardware memory space called a **Device**:
- `device('cpu')`: Standard system RAM (Random Access Memory).
- `device('cuda')`: NVIDIA GPU Video RAM (VRAM), powered by CUDA.
- `device('mps')`: Apple Silicon GPU RAM (Metal Performance Shaders on M1/M2/M3 chips).

The golden rule of PyTorch hardware is: **Tensors on different devices CANNOT interact.**
If you try to add a CPU tensor to a GPU tensor:
```python
x_cpu + y_gpu  # 💥 Crash! RuntimeError: Expected all tensors to be on the same device!
```

Furthermore, sending data across the PCIe bus from CPU RAM to GPU VRAM takes time. You should always transfer data in **large batches**, rather than moving numbers one by one!

---

### 🌍 Real-Life Example

**Scenario:** Building production AI code that runs seamlessly on a researcher's MacBook, an engineer's laptop without a GPU, and a cloud server cluster with 8x NVIDIA H100 GPUs.

| User Hardware | Available Accelerator | PyTorch Auto-Detection | Device Assigned |
|:---|:---|:---|:---|
| Cloud Training Server | NVIDIA H100 GPU | `torch.cuda.is_available() == True` | `torch.device('cuda')` |
| Developer MacBook | Apple Silicon M3 Max | `torch.backends.mps.is_available() == True` | `torch.device('mps')` |
| Standard PC Laptop | Intel / AMD CPU | All accelerators `False` | `torch.device('cpu')` |

**Why this matters:** Professional AI code never hardcodes `.cuda()`. It uses dynamic device configuration so the exact same script runs anywhere in the world without crashing!

---

### 📖 How It Works (Step by Step)

#### The Canonical Device Setup Pattern
```python
# Universal device selector pattern used in all production AI systems
if torch.cuda.is_available():
    device = torch.device('cuda')
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

print(f"Executing pipeline on: {device}")
```

#### Moving Tensors and Models
1. **Moving a Tensor:** `x_gpu = x.to(device)` (returns a new tensor reference on the target device).
2. **Moving an `nn.Module` Network:** `model.to(device)` (moves all internal layer weights and biases in-place!).
3. **Retrieving Predictions back to CPU:** `predictions = model(x.to(device)).cpu()`

---

### 🔑 Key Concepts

| Method | Behavior | Return Value |
|:---|:---|:---|
| `tensor.to(device)` | Allocates VRAM and copies data over PCIe bus. | **Returns new tensor** (does NOT modify in-place). |
| `model.to(device)` | Moves all internal weights/biases to target device. | **Modifies in-place** and returns self. |
| `tensor.is_cuda` | Boolean flag checking if tensor is in GPU VRAM. | `True` or `False`. |

---

### 🗺️ Visual Overview

```
THE PCIE BUS DATA TRANSFER CHOKEPOINT:

  ┌─────────────────────────┐                            ┌─────────────────────────┐
  │     CPU System RAM      │                            │     GPU Video RAM       │
  │                         │       PCIe Express Bus     │                         │
  │  x_cpu = torch.randn()  │ ──── .to('cuda') ────────► │  x_gpu (Ready for CUDA) │
  │                         │ ◄─── .to('cpu') ────────── │  y_gpu = model(x_gpu)   │
  └─────────────────────────┘                            └─────────────────────────┘
  Transferring small tensors repeatedly causes massive PCIe bus bottlenecks!
```

---

### 🚫 Common Misconceptions

| ❌ Myth | ✅ Reality |
|:---|:---|
| *"Calling `tensor.to('cuda')` moves the tensor in-place like a list."* | **A very common bug!** `tensor.to('cuda')` does NOT modify the original tensor. It returns a **new tensor** on the GPU. You must capture the return value: `tensor = tensor.to('cuda')`! (In contrast, `model.to('cuda')` DOES modify the model in-place). |

---

### ✅ When to Use This

- Always write `device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')` at the top of every training script.

---

### 📊 Section Quick Reference

| Command | Action |
|:---|:---|
| `torch.cuda.is_available()` | Checks if NVIDIA GPU driver is functional |
| `torch.cuda.device_count()` | Returns number of physical GPUs |
| `x = x.to(device)` | Moves tensor to active accelerator |
| `model = model.to(device)` | Moves entire neural network to accelerator |

---


In [4]:
# ✅ Code Demo: Universal Device Detection & Robust Multi-Platform Device Routing
# Concept: Dynamic Hardware Detection, Safe Tensor Migration, and Error Handling
# -------------------------------------------------------------------------------

import os
import torch

# Ensure local asset path exists
assets_dir = 'assets' if os.path.exists('assets') else 'Module-06-PyTorch-Foundations/assets'
os.makedirs(assets_dir, exist_ok=True)

# 1. Professional Multi-Platform Device Selection Pattern
def get_best_available_device():
    if torch.cuda.is_available():
        return torch.device('cuda')
    elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        return torch.device('mps')
    else:
        return torch.device('cpu')

active_device = get_best_available_device()

print("=" * 65)
print("PYTORCH ACCELERATOR HARDWARE INSPECTION")
print("=" * 65)
print(f"Active Selected Device:      {active_device}")
print(f"NVIDIA CUDA Available:       {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA Device Count:           {torch.cuda.device_count()}")
    print(f"Current GPU Name:            {torch.cuda.get_device_name(0)}")
print(f"Apple MPS Available:         {hasattr(torch.backends, 'mps') and torch.backends.mps.is_available()}")
print("-" * 65)

# 2. Moving Tensors to Device Safely
x = torch.randn(3, 3)
x_on_device = x.to(active_device)

print(f"Tensor x Device:             {x.device}")
print(f"Tensor x_on_device Device:   {x_on_device.device}")
print(f"Is x on CUDA?                {x.is_cuda}")
print(f"Is x_on_device on CUDA?      {x_on_device.is_cuda}")

# 3. Demonstrating Safe Fallback Behavior
print("\nDemonstrating Safe Cross-Device Operations:")
y_cpu = torch.ones(3, 3, device='cpu')
# Move to matching device before computation
y_matched = y_cpu.to(x_on_device.device)
z_result = x_on_device + y_matched
print(f"Resulting computation device: {z_result.device} (Clean execution with no crashes!)")
print("=" * 65)


PYTORCH ACCELERATOR HARDWARE INSPECTION
Active Selected Device:      cpu
NVIDIA CUDA Available:       False
Apple MPS Available:         False
-----------------------------------------------------------------
Tensor x Device:             cpu
Tensor x_on_device Device:   cpu
Is x on CUDA?                False
Is x_on_device on CUDA?      False

Demonstrating Safe Cross-Device Operations:
Resulting computation device: cpu (Clean execution with no crashes!)


---

## 5. Dimension Surgery — View vs Reshape, Squeeze, & Unsqueeze

---

### 🗣️ In Simple English

In deep learning, you spend $50\%$ of your time writing model code, and the other $50\%$ fixing dimension mismatch errors like:
`RuntimeError: Expected 4D tensor, but got 3D tensor`!

To fix dimensions effortlessly, you need three surgical tools:
1. **Reshaping (`.view()` and `.reshape()`):** Changing the dimensions without changing the total count of numbers. Example: Flattening a $28 \times 28$ image into a flat $784$-element vector.
2. **Unsqueeze (`.unsqueeze(dim)`):** Adding a fake "dummy" dimension of size $1$. Example: Turning a single image `(3, 28, 28)` into a batch of one image `(1, 3, 28, 28)` so a neural network can accept it.
3. **Squeeze (`.squeeze()`):** Stripping away all unnecessary dimensions of size $1$. Example: Collapsing `(1, 10, 1)` into a clean vector of `(10,)`.

The critical technical difference is: **`.view()` vs `.reshape()`**:
- `.view()` is blazing fast because it **guarantees zero-copy memory reuse**, but it requires the tensor to be **contiguous** in memory.
- `.reshape()` is safer: it uses `.view()` if possible, but automatically makes a copy in memory if the tensor is non-contiguous!

---

### 🌍 Real-Life Example

**Scenario:** Passing a single photo captured from your webcam into a ResNet vision model expecting a batch dimension.

```
Your Webcam Image Shape:      (3, 224, 224)          <-- Missing Batch Dimension!
ResNet Input Requirement:    (Batch, 3, 224, 224)   <-- Requires 4D Tensor!

Solution:  img_batch = img.unsqueeze(0)             --> Shape: (1, 3, 224, 224) ✅
```

---

### 📖 How It Works (Step by Step)

#### A. The `-1` Automatic Dimension Trick
When reshaping, you can replace **one single dimension** with `-1`. PyTorch will automatically calculate the missing dimension size for you!
- Original shape: `(128, 3, 28, 28)` (Total elements: $128 \times 3 \times 28 \times 28 = 301,056$).
- Flatten features per sample: `x.view(128, -1)`.
- PyTorch calculates: $301,056 / 128 = 2,352$.
- Result shape: `(128, 2352)`!

#### B. Squeeze and Unsqueeze Mechanics
- `t = torch.randn(1, 3, 1, 28)`
- `t.squeeze()`: Removes ALL dimensions of size $1 \implies (3, 28)$.
- `t.squeeze(0)`: Only removes dimension 0 if it is size $1 \implies (3, 1, 28)$.
- `t = torch.randn(28, 28)`
- `t.unsqueeze(0)`: Inserts size 1 at position $0 \implies (1, 28, 28)$.
- `t.unsqueeze(-1)`: Inserts size 1 at the end $\implies (28, 28, 1)$.

---

### 🔑 Key Concepts

| Method | What It Does | Memory Effect |
|:---|:---|:---|
| `t.view(*shape)` | Reshapes tensor based strictly on strides. | Zero-copy guaranteed; fails if non-contiguous. |
| `t.reshape(*shape)` | Reshapes tensor; calls `.contiguous().view()` if needed. | Zero-copy if possible; copies memory if non-contiguous. |
| `t.contiguous()` | Rearranges memory slots into sequential flat order. | Allocates new memory buffer if strides were transposed. |
| `t.unsqueeze(dim)` | Adds a dimension of size 1 at index `dim`. | Zero-copy view. |
| `t.squeeze(dim)` | Removes dimension of size 1 at index `dim`. | Zero-copy view. |

---

### 🗺️ Visual Overview

```
SQUEEZE AND UNSQUEEZE SURGERY:

Unsqueeze(dim=0): Adds batch axis
  Matrix (3, 3) ───────────────► 3D Tensor (1, 3, 3) [A single batch containing 1 matrix]

Squeeze(): Strips redundant singletons
  4D Tensor (1, 10, 1, 1) ──────► 1D Vector (10,)
```

---

### 🚫 Common Misconceptions

| ❌ Myth | ✅ Reality |
|:---|:---|
| *"Why did `x.t().view(-1)` throw `RuntimeError: view size is not compatible with input tensor's size and stride`?"* | Transposing a matrix swaps its strides without moving memory. The tensor is now **non-contiguous**! Because `.view()` requires elements to be adjacent in physical RAM, you must call `x.t().contiguous().view(-1)` or simply use `x.t().reshape(-1)`! |

---

### ✅ When to Use This

- Use `.reshape(B, -1)` when flattening convolutional maps before fully connected linear layers.
- Use `.unsqueeze(0)` whenever running inference on a single image.

---

### 📊 Section Quick Reference

| Method | Input Shape | Output Shape |
|:---|:---:|:---:|
| `t.view(B, -1)` | `(B, C, H, W)` | `(B, C * H * W)` (Flattened) |
| `t.unsqueeze(0)` | `(C, H, W)` | `(1, C, H, W)` (Batch added) |
| `t.squeeze()` | `(1, C, 1, H)` | `(C, H)` (Singletons removed) |

---


In [5]:
# ✅ Code Demo: Dimension Surgery — Reshaping, Flattening, Squeezing, and Contiguity
# Concept: .view() vs .reshape(), Squeeze/Unsqueeze, and Resolving Contiguity Errors
# ---------------------------------------------------------------------------------

import os
import torch

# Ensure local asset path exists
assets_dir = 'assets' if os.path.exists('assets') else 'Module-06-PyTorch-Foundations/assets'
os.makedirs(assets_dir, exist_ok=True)

# 1. Simulating a Batch of 4 Grayscale MNIST Images: (Batch=4, Channels=1, Height=28, Width=28)
mnist_batch = torch.randn(4, 1, 28, 28)
print("=" * 65)
print("DIMENSION SURGERY: RESHAPING & FLATTENING")
print("=" * 65)
print(f"Original Batch Shape: {mnist_batch.shape}")

# Flatten spatial features using -1 automatic dimension trick
flattened = mnist_batch.view(4, -1)
print(f"Flattened via .view(4, -1): {flattened.shape} (4 samples x 784 features)")

# 2. Squeeze & Unsqueeze Surgery
single_image = torch.randn(28, 28) # 2D image without batch or channel
print("\nSQUEEZE AND UNSQUEEZE IN ACTION:")
print(f"  Raw Image Shape:                    {single_image.shape}")

# Add channel dimension (1, 28, 28)
with_channel = single_image.unsqueeze(0)
print(f"  After unsqueeze(0) [Channel added]: {with_channel.shape}")

# Add batch dimension (1, 1, 28, 28) -> Ready for Neural Network!
with_batch = with_channel.unsqueeze(0)
print(f"  After unsqueeze(0) [Batch added]:   {with_batch.shape} (Ready for Model!)")

# Squeeze all redundant singletons back to 2D
squeezed_back = with_batch.squeeze()
print(f"  After .squeeze() [Stripped]:        {squeezed_back.shape}")
print("-" * 65)

# 3. The Classic Contiguity Gotcha & Resolution
print("THE CONTIGUITY GOTCHA: .view() VS .reshape()")
print("-" * 65)
matrix = torch.tensor([[1, 2, 3], [4, 5, 6]])  # Shape (2, 3)
transposed = matrix.t()                        # Shape (3, 2), non-contiguous!

print(f"Original Matrix is_contiguous:   {matrix.is_contiguous()}")
print(f"Transposed Matrix is_contiguous: {transposed.is_contiguous()}")

# Attempting .view() on non-contiguous tensor
try:
    _ = transposed.view(-1)
except RuntimeError as e:
    print(f"\nCaught Expected Error calling .view() on transposed tensor:")
    print(f"  -> {str(e)[:70]}...")

# Clean resolution using .reshape() or .contiguous().view()
resolved_reshape = transposed.reshape(-1)
resolved_contiguous = transposed.contiguous().view(-1)
print(f"\nClean Resolution via .reshape():          {resolved_reshape}")
print(f"Clean Resolution via .contiguous().view(): {resolved_contiguous}")
print("=" * 65)


DIMENSION SURGERY: RESHAPING & FLATTENING
Original Batch Shape: torch.Size([4, 1, 28, 28])
Flattened via .view(4, -1): torch.Size([4, 784]) (4 samples x 784 features)

SQUEEZE AND UNSQUEEZE IN ACTION:
  Raw Image Shape:                    torch.Size([28, 28])
  After unsqueeze(0) [Channel added]: torch.Size([1, 28, 28])
  After unsqueeze(0) [Batch added]:   torch.Size([1, 1, 28, 28]) (Ready for Model!)
  After .squeeze() [Stripped]:        torch.Size([28, 28])
-----------------------------------------------------------------
THE CONTIGUITY GOTCHA: .view() VS .reshape()
-----------------------------------------------------------------
Original Matrix is_contiguous:   True
Transposed Matrix is_contiguous: False

Caught Expected Error calling .view() on transposed tensor:
  -> view size is not compatible with input tensor's size and stride (at le...

Clean Resolution via .reshape():          tensor([1, 4, 2, 5, 3, 6])
Clean Resolution via .contiguous().view(): tensor([1, 4, 2, 5, 3, 6])


---

## 📝 Chapter Summary

| Concept | Plain English Definition | Key Syntax / Function | Crucial Takeaway |
|:---|:---|:---|:---|
| **Tensor** | Multi-dimensional numerical array with GPU support | `torch.Tensor` | The atomic data structure of all PyTorch computation. |
| **Strides** | Memory jump distances for each tensor axis | `t.stride()` | Allows instant $O(1)$ zero-copy transpositions and views. |
| **NumPy Bridge** | Shared-memory array conversion | `torch.from_numpy(arr)` | Zero-copy; mutating one modifies the other in-place. |
| **Matmul Operator** | True linear algebra matrix multiplication | `A @ B` or `torch.matmul(A, B)` | Not element-wise `*`; inner dimensions must match. |
| **Device Migration** | Moving tensors between CPU and GPU | `t = t.to(device)` | Tensors on different devices cannot interact. |
| **Reshape vs View** | Dimension surgery | `t.reshape(-1)` vs `t.view(-1)` | `.view()` requires contiguity; `.reshape()` handles copies automatically. |
| **Squeeze / Unsqueeze**| Adding or removing singleton dimensions | `t.unsqueeze(dim)` / `t.squeeze()` | Prepares image tensors for batch requirements. |

---


## 🔗 Resources & Further Reading

| Resource | Type | Why Read It |
|:---|:---|:---|
| **[PyTorch Official Documentation: Tensors](https://pytorch.org/docs/stable/tensors.html)** | Official Docs | Comprehensive API reference for all 100+ tensor operations. |
| **[Deep Learning with PyTorch: A 60 Minute Blitz](https://pytorch.org/tutorials/beginner/deep_learning_60min_blitz.html)** | Landmark Tutorial | Soumith Chintala's classic fast-paced introduction to PyTorch fundamentals. |
| **[PyTorch Internals: ezyang's Blog](http://blog.ezyang.com/2019/05/pytorch-internals/)** | Deep Technical Architecture | Edward Z. Yang's authoritative breakdown of how PyTorch manages C++ ATen storage, strides, and dispatchers. |

---


## ➡️ Next Chapter: Chapter 6.2 — Autograd: PyTorch's Automatic Differentiation

Now that we master tensors, we unlock PyTorch's computational magic:
- **The Autograd Engine:** How `requires_grad=True` turns tensors into dynamic graph nodes.
- **The Graph Tracker:** `grad_fn` and the directed acyclic graph built automatically during the forward pass.
- **The Backward Call:** Calling `.backward()` and inspecting `.grad`.
- **Detaching & Inference:** `with torch.no_grad():` and `.detach()` to disable gradient tracking and save GPU memory.

---
*Made with ❤️ | Zero to Hero — AI Engineer Learning Path*
